In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('/Users/emersonfrost/Desktop/caml_60days_post.csv')
print(df.shape)
df.head(5)

(130791, 18)


,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,ndti,ndvi,ndwi,red,red_edge,sample_date,satellite_date,.geo
0,0_20200703T170901_20200703T171739_T14SQJ,0.232479,0.272720,0.133473,0.900524,6.061,0.169426,39.141931,-95.479569,0.277230,-0.053784,0.104220,-0.157098,0.188913,0.334219,2020-06-29,2020-07-03,"{""type"":""MultiPoint"",""coordinates"":[]}"
1,0_20200703T170901_20200703T171739_T15STD,0.246377,0.273991,0.157615,0.851792,6.061,0.187010,39.141931,-95.479569,0.253915,-0.037214,0.100385,-0.137050,0.201706,0.339429,2020-06-29,2020-07-03,"{""type"":""MultiPoint"",""coordinates"":[]}"
2,0_20200708T170849_20200708T171639_T14SQJ,0.184233,0.262298,0.097692,0.149765,6.061,0.139081,39.141931,-95.479569,0.091999,-0.047473,0.080031,-0.126718,0.158255,0.199905,2020-06-29,2020-07-08,"{""type"":""MultiPoint"",""coordinates"":[]}"
3,0_20200708T170849_20200708T171639_T15STD,0.183615,0.261502,0.098261,0.185295,6.061,0.139267,39.141931,-95.479569,0.093275,-0.046803,0.079792,-0.126165,0.158236,0.200786,2020-06-29,2020-07-08,"{""type"":""MultiPoint"",""coordinates"":[]}"
4,0_20200713T170901_20200713T171937_T14SQJ,0.182510,0.258955,0.088746,0.001473,6.061,0.134364,39.141931,-95.479569,0.140973,-0.059787,0.088269,-0.147829,0.155458,0.210817,2020-06-29,2020-07-13,"{""type"":""MultiPoint"",""coordinates"":[]}"


In [6]:
df['cyanobacteria_abundance'].nunique()

6297

In [7]:
#dropping null / inconsistent values 
print(df.shape)
df = df[(df['green'] != -999) & (df['blue'] != -999) & (df['SWIR'] != -999) & (df['NIR'] != -999)]
print(df.shape)

(130791, 18)
(130552, 18)


In [8]:
#now creating cloud score barriers

#30%
df_under30 = df[df['cloud_score'] <= 0.3]
print("Under 30%: ", df_under30.shape)
#50% 
df_under50 = df[df['cloud_score'] <= 0.5]
print("Under 50%: ", df_under50.shape)
#70%
df_under70 = df[df['cloud_score'] <= 0.7]
print("Under 70%: ", df_under70.shape)

Under 30%:  (66310, 18)
Under 50%:  (80748, 18)
Under 70%:  (92847, 18)


In [9]:
#percentages
print(df_under30.shape[0] / df.shape[0])
print(df_under50.shape[0] / df.shape[0])
print(df_under70.shape[0] / df.shape[0])

0.5079202156994914
0.618512163735523
0.7111878791592622


In [10]:
#removing duplicates ... keep the version that has a lower cloud score
df_under30_clean = (df_under30.sort_values('cloud_score', ascending = True).drop_duplicates(subset = ['cyanobacteria_abundance','sample_date', 'satellite_date']))
df_under50_clean = (df_under50.sort_values('cloud_score', ascending = True).drop_duplicates(subset = ['cyanobacteria_abundance','sample_date', 'satellite_date']))
df_under70_clean = (df_under70.sort_values('cloud_score', ascending = True).drop_duplicates(subset = ['cyanobacteria_abundance','sample_date', 'satellite_date']))


In [11]:
print(df_under30_clean.shape)
print(df_under50_clean.shape)
print(df_under70_clean.shape)

(53717, 18)
(64762, 18)
(73822, 18)


In [18]:
#limiting to 5 specific time points after 
dfs = [df_under30_clean, df_under50_clean, df_under70_clean]
for cur_df in dfs:
    #convert to datetime objects
    cur_df['sample_date'] = pd.to_datetime(cur_df['sample_date'])
    cur_df['satellite_date'] = pd.to_datetime(cur_df['satellite_date'])
    
    #add column that has diff_days between sample_date and sat_date
    cur_df['tot_diff_days'] = (cur_df['satellite_date'] - cur_df['sample_date']).dt.days

print(df_under30_clean.columns)



Index(['system:index', 'NIR', 'SWIR', 'blue', 'cloud_score',
       'cyanobacteria_abundance', 'green', 'latitude', 'longitude', 'ndci',
       'ndti', 'ndvi', 'ndwi', 'red', 'red_edge', 'sample_date',
       'satellite_date', '.geo', 'tot_diff_days'],
      dtype='object')


In [19]:
df_under30_clean.head(5)

,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,ndti,ndvi,ndwi,red,red_edge,sample_date,satellite_date,.geo,tot_diff_days
82575,5676_20210914T185929_20210914T191123_T10TFL,0.036914,0.052851,0.021093,0.0,2702.500000,0.025146,41.049098,-121.762001,0.795851,0.161299,0.342186,-0.193357,0.018243,0.159551,2021-08-23,2021-09-14,"{""type"":""MultiPoint"",""coordinates"":[]}",22
38771,2370_20171023T182409_20171023T182649_T12SUG,0.000000,0.002615,0.009567,0.0,11.712000,0.017497,37.522430,-112.771798,-0.993958,0.892017,-0.993958,1.000000,0.001020,0.000000,2017-09-06,2017-10-23,"{""type"":""MultiPoint"",""coordinates"":[]}",47
112278,7828_20210907T172859_20210907T173213_T14TLP,0.098599,0.241983,0.041572,0.0,689.493110,0.061126,43.769466,-100.268631,0.562118,-0.013499,0.221880,-0.234843,0.062949,0.224788,2021-08-24,2021-09-07,"{""type"":""MultiPoint"",""coordinates"":[]}",14
127412,8947_20180709T185019_20180709T185935_T10TEK,0.106604,0.160684,0.044539,0.0,7720.000000,0.068973,39.753301,-121.971000,0.699173,0.137362,0.339502,-0.216213,0.053305,0.313730,2018-05-16,2018-07-09,"{""type"":""MultiPoint"",""coordinates"":[]}",54
21263,1301_20170928T182131_20170928T182608_T12TVQ,0.069945,0.078710,0.018209,0.0,71.259201,0.038488,44.601953,-112.058699,0.408164,0.044378,0.326215,-0.287486,0.036668,0.092000,2017-09-06,2017-09-28,"{""type"":""MultiPoint"",""coordinates"":[]}",22


In [55]:
#selecting the four points that we want to keep
def select_rows_unique(group):
    group = group.sort_values('tot_diff_days').copy()
    selected_rows = []
    used_idx = set()

    #one week, two weeks, one month
    targets = [7, 14, 30]

    # pick closest to each target without reuse
    for t in targets:
        remaining = group.loc[~group.index.isin(used_idx)]
        if remaining.empty:
            break

        idx = (remaining['tot_diff_days'] - t).abs().idxmin()
        row = group.loc[idx].copy()
        row['target_days'] = t
        row['days_off'] = row['tot_diff_days'] - t

        selected_rows.append(row)
        used_idx.add(idx)

    # closest to two months
    remaining = group.loc[~group.index.isin(used_idx)]
    if not remaining.empty:
        idx_max = remaining['tot_diff_days'].idxmax()
        row_max = group.loc[idx_max].copy()
        row_max['target_days'] = 60
        row_max['days_off'] = row_max['tot_diff_days'] - 60

        selected_rows.append(row_max)

    return pd.DataFrame(selected_rows)

In [56]:
print(df_under30_clean.shape)
df_one = (
    df_under30_clean
    .groupby(['sample_date', 'cyanobacteria_abundance'], group_keys=False)
    .apply(select_rows_unique)
    .reset_index(drop=True)
)
print(df_one.shape)

(53717, 19)
(26737, 21)


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_85694/3415155298.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_rows_unique)


In [57]:
df_one = (
    df_one
    .groupby(['sample_date', 'cyanobacteria_abundance'])
    .filter(lambda x: len(x) == 4)
    .reset_index(drop=True)
)

In [58]:
df_one.shape

(23108, 21)

In [59]:
#select two examples to visualize
df_one.sort_values(['sample_date', 'cyanobacteria_abundance'])
df_one.head(16)

,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,ndvi,ndwi,red,red_edge,sample_date,satellite_date,.geo,tot_diff_days,target_days,days_off
0,8282_20160113T184732_20160113T184730_T10SGF,0.156475,0.212725,0.079949,0.290662,5545.0,0.114887,36.748901,-120.512998,0.136718,...,0.075642,-0.154208,0.134779,0.177262,2016-01-08,2016-01-13,"{""type"":""MultiPoint"",""coordinates"":[]}",5,7,-2
1,8282_20160126T185642_20160126T185741_T10SGF,0.204080,0.278726,0.104335,0.053516,5545.0,0.152208,36.748901,-120.512998,0.127492,...,0.078338,-0.151314,0.176705,0.225079,2016-01-08,2016-01-26,"{""type"":""MultiPoint"",""coordinates"":[]}",18,14,4
2,8282_20160205T190342_20160205T190345_T10SGF,0.231652,0.321285,0.122737,0.147210,5545.0,0.174651,36.748901,-120.512998,0.111743,...,0.065413,-0.144294,0.204867,0.254099,2016-01-08,2016-02-05,"{""type"":""MultiPoint"",""coordinates"":[]}",28,30,-2
3,8282_20160225T185342_20160225T185338_T10SGF,0.207113,0.301482,0.107721,0.000007,5545.0,0.153597,36.748901,-120.512998,0.115962,...,0.054328,-0.154503,0.187511,0.234823,2016-01-08,2016-02-25,"{""type"":""MultiPoint"",""coordinates"":[]}",48,60,-12
4,8283_20160113T184732_20160113T184730_T10SGF,0.134736,0.184039,0.067658,0.290662,5790.0,0.094283,36.651902,-120.632003,0.329047,...,0.156312,-0.180758,0.099874,0.201455,2016-01-08,2016-01-13,"{""type"":""MultiPoint"",""coordinates"":[]}",5,7,-2
5,8283_20160126T185642_20160126T185741_T10SGF,0.156168,0.210376,0.076890,0.053516,5790.0,0.109932,36.651902,-120.632003,0.351967,...,0.176350,-0.193104,0.115893,0.237234,2016-01-08,2016-01-26,"{""type"":""MultiPoint"",""coordinates"":[]}",18,14,4
6,8283_20160205T190342_20160205T190345_T10SGF,0.162451,0.212995,0.087617,0.147210,5790.0,0.118639,36.651902,-120.632003,0.351164,...,0.164817,-0.170485,0.121395,0.249718,2016-01-08,2016-02-05,"{""type"":""MultiPoint"",""coordinates"":[]}",28,30,-2
7,8283_20160225T185342_20160225T185338_T10SGF,0.172988,0.241258,0.084965,0.000007,5790.0,0.119216,36.651902,-120.632003,0.359918,...,0.161736,-0.193508,0.129069,0.270625,2016-01-08,2016-02-25,"{""type"":""MultiPoint"",""coordinates"":[]}",48,60,-12
8,8776_20160206T183312_20160206T183306_T11SMT,0.116573,0.202712,0.070986,0.000815,3824.5,0.090925,33.474200,-117.137998,0.298811,...,0.124935,-0.271286,0.106040,0.157637,2016-01-21,2016-02-06,"{""type"":""MultiPoint"",""coordinates"":[]}",16,7,9
9,8776_20160209T184442_20160209T184443_T11SMT,0.115190,0.183930,0.072365,0.000432,3824.5,0.091460,33.474200,-117.137998,0.272391,...,0.118438,-0.230684,0.104561,0.147881,2016-01-21,2016-02-09,"{""type"":""MultiPoint"",""coordinates"":[]}",19,14,5


In [60]:
#number of unique
df_one['cyanobacteria_abundance'].nunique()

4810

In [61]:
#looks good. Now apply to the other two examples
#under 50 sample
print(df_under50_clean.shape)
df_two = (
    df_under50_clean
    .groupby(['sample_date', 'cyanobacteria_abundance'], group_keys=False)
    .apply(select_rows_unique)
    .reset_index(drop=True)
)
print(df_two.shape)
df_two = (
    df_two
    .groupby(['sample_date', 'cyanobacteria_abundance'])
    .filter(lambda x: len(x) == 4)
    .reset_index(drop=True)
)
print(df_two.shape)
print(df_two['cyanobacteria_abundance'].nunique())



(64762, 19)
(28220, 21)
(25900, 21)
5334


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_85694/3042312006.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_rows_unique)


In [62]:
#under 70 sample
print(df_under70_clean.shape)
df_three = (
    df_under70_clean
    .groupby(['sample_date', 'cyanobacteria_abundance'], group_keys=False)
    .apply(select_rows_unique)
    .reset_index(drop=True)
)
print(df_three.shape)
df_three = (
    df_three
    .groupby(['sample_date', 'cyanobacteria_abundance'])
    .filter(lambda x: len(x) == 4)
    .reset_index(drop=True)
)
print(df_three.shape)
print(df_three['cyanobacteria_abundance'].nunique())

(73822, 19)
(28903, 21)
(26908, 21)
5539


/var/folders/9z/ltjvr1610w9g431sszz26tr40000gn/T/ipykernel_85694/2911565308.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(select_rows_unique)


In [63]:
#now trying to match up with the 3 samples for time series points before sample date
before_1 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/timedf_30_7.csv')
before_2 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/timedf_50_8.csv')
before_3 = pd.read_csv('/Users/emersonfrost/Desktop/MLfCC/mlcc-final-project/timedf_70_9.csv')

#match up unique cases by 'abun' and 'sample_date' --> then we create training + test set for each of the three cases
#using the less than 70% filter
uni_samp = before_3[['cyanobacteria_abundance', 'sample_date']].drop_duplicates()
print(uni_samp.shape)

uni_here = df_three[['cyanobacteria_abundance', 'sample_date']].drop_duplicates()
print(uni_here.shape)

key_cols = ['cyanobacteria_abundance', 'sample_date']
uni_samp['sample_date'] = pd.to_datetime(uni_samp['sample_date'])
uni_here['sample_date'] = pd.to_datetime(uni_here['sample_date'])
common_keys = uni_samp.merge(uni_here, on=key_cols, how='inner')

print(common_keys.shape)


(5640, 2)
(6727, 2)
(5378, 2)


In [64]:
#third dataframe combine --> 70% cloud coverage for both test and train set... 9 sample input
#change to datetime
before_3['sample_date'] = pd.to_datetime(before_3['sample_date'])
before_3['satellite_date'] = pd.to_datetime(before_3['satellite_date'])

filtered_train_df1 = before_3.merge(common_keys, on=key_cols, how='inner')
filtered_test_df1 = df_three.merge(common_keys, on=key_cols, how='inner')

print(filtered_train_df1.shape)
print(filtered_train_df1[['cyanobacteria_abundance', 'sample_date']].drop_duplicates().shape)

print(filtered_test_df1.shape)
print(filtered_test_df1[['cyanobacteria_abundance', 'sample_date']].drop_duplicates().shape) 

(48402, 22)
(5378, 2)
(21512, 21)
(5378, 2)


In [65]:
#second dataframe combine --> 50% cloud coverage for both train and test set ... 8 sample input
uni_samp = before_2[['cyanobacteria_abundance', 'sample_date']].drop_duplicates()
print(uni_samp.shape)

uni_here = df_two[['cyanobacteria_abundance', 'sample_date']].drop_duplicates()
print(uni_here.shape)

key_cols = ['cyanobacteria_abundance', 'sample_date']
uni_samp['sample_date'] = pd.to_datetime(uni_samp['sample_date'])
uni_here['sample_date'] = pd.to_datetime(uni_here['sample_date'])
common_keys = uni_samp.merge(uni_here, on=key_cols, how='inner')

print(common_keys.shape)

before_2['sample_date'] = pd.to_datetime(before_2['sample_date'])
before_2['satellite_date'] = pd.to_datetime(before_2['satellite_date'])

filtered_train_df2 = before_2.merge(common_keys, on=key_cols, how='inner')
filtered_test_df2 = df_two.merge(common_keys, on=key_cols, how='inner')

print(filtered_train_df2.shape)
print(filtered_train_df2[['cyanobacteria_abundance', 'sample_date']].drop_duplicates().shape)

print(filtered_test_df2.shape)
print(filtered_test_df2[['cyanobacteria_abundance', 'sample_date']].drop_duplicates().shape) 

(5428, 2)
(6475, 2)
(5130, 2)
(41040, 22)
(5130, 2)
(20520, 21)
(5130, 2)


In [66]:
#now do the same for the first dataframe --> under 30% w/ 7 time steps
uni_samp = before_1[['cyanobacteria_abundance', 'sample_date']].drop_duplicates()
print(uni_samp.shape)

uni_here = df_one[['cyanobacteria_abundance', 'sample_date']].drop_duplicates()
print(uni_here.shape)

key_cols = ['cyanobacteria_abundance', 'sample_date']
uni_samp['sample_date'] = pd.to_datetime(uni_samp['sample_date'])
uni_here['sample_date'] = pd.to_datetime(uni_here['sample_date'])
common_keys = uni_samp.merge(uni_here, on=key_cols, how='inner')

print(common_keys.shape)

before_1['sample_date'] = pd.to_datetime(before_1['sample_date'])
before_1['satellite_date'] = pd.to_datetime(before_1['satellite_date'])

filtered_train_df3 = before_1.merge(common_keys, on=key_cols, how='inner')
filtered_test_df3 = df_one.merge(common_keys, on=key_cols, how='inner')

print(filtered_train_df3.shape)
print(filtered_train_df3[['cyanobacteria_abundance', 'sample_date']].drop_duplicates().shape)

print(filtered_test_df3.shape)
print(filtered_test_df3[['cyanobacteria_abundance', 'sample_date']].drop_duplicates().shape)

(4804, 2)
(5777, 2)
(4357, 2)
(30499, 22)
(4357, 2)
(17428, 21)
(4357, 2)


In [67]:
filtered_test_df3.head(10)

,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,ndvi,ndwi,red,red_edge,sample_date,satellite_date,.geo,tot_diff_days,target_days,days_off
0,8776_20160206T183312_20160206T183306_T11SMT,0.116573,0.202712,0.070986,0.000815,3824.500000,0.090925,33.474200,-117.137998,0.298811,...,0.124935,-0.271286,0.106040,0.157637,2016-01-21,2016-02-06,"{""type"":""MultiPoint"",""coordinates"":[]}",16,7,9
1,8776_20160209T184442_20160209T184443_T11SMT,0.115190,0.183930,0.072365,0.000432,3824.500000,0.091460,33.474200,-117.137998,0.272391,...,0.118438,-0.230684,0.104561,0.147881,2016-01-21,2016-02-09,"{""type"":""MultiPoint"",""coordinates"":[]}",19,14,5
2,8776_20160219T184442_20160219T184442_T11SMT,0.123797,0.184201,0.080175,0.017917,3824.500000,0.099677,33.474200,-117.137998,0.289272,...,0.129392,-0.208424,0.110635,0.163088,2016-01-21,2016-02-19,"{""type"":""MultiPoint"",""coordinates"":[]}",29,30,-1
3,8776_20160317T182242_20160317T183021_T11SMT,0.135107,0.205891,0.076413,0.000281,3824.500000,0.103360,33.474200,-117.137998,0.485052,...,0.261211,-0.213258,0.098753,0.232921,2016-01-21,2016-03-17,"{""type"":""MultiPoint"",""coordinates"":[]}",56,60,-4
4,8777_20160206T183312_20160206T183306_T11SMT,0.093039,0.218421,0.019219,0.000815,5943.000000,0.047272,33.477901,-117.141998,0.479036,...,0.225797,-0.345080,0.061234,0.180323,2016-01-21,2016-02-06,"{""type"":""MultiPoint"",""coordinates"":[]}",16,7,9
5,8777_20160209T184442_20160209T184443_T11SMT,0.086222,0.193044,0.025243,0.000432,5943.000000,0.045851,33.477901,-117.141998,0.462960,...,0.201753,-0.319615,0.058793,0.165607,2016-01-21,2016-02-09,"{""type"":""MultiPoint"",""coordinates"":[]}",19,14,5
6,8777_20160219T184442_20160219T184442_T11SMT,0.085882,0.181212,0.025278,0.017917,5943.000000,0.046271,33.477901,-117.141998,0.500021,...,0.226834,-0.312595,0.055818,0.174593,2016-01-21,2016-02-19,"{""type"":""MultiPoint"",""coordinates"":[]}",29,30,-1
7,8777_20160317T182242_20160317T183021_T11SMT,0.102755,0.206258,0.025263,0.000281,5943.000000,0.058008,33.477901,-117.141998,0.668023,...,0.360712,-0.285972,0.050129,0.260596,2016-01-21,2016-03-17,"{""type"":""MultiPoint"",""coordinates"":[]}",56,60,-4
8,7117_20160415T190352_20160415T190349_T10SFG,0.175815,0.281559,0.093093,0.002653,1201.413452,0.124639,37.458299,-120.967002,0.298554,...,0.119865,-0.175910,0.141039,0.255824,2016-04-12,2016-04-15,"{""type"":""MultiPoint"",""coordinates"":[]}",3,7,-4
9,7117_20160425T184922_20160425T185405_T10SFG,0.192180,0.312428,0.109277,0.121317,1201.413452,0.138950,37.458299,-120.967002,0.224341,...,0.069412,-0.164012,0.168919,0.263846,2016-04-12,2016-04-25,"{""type"":""MultiPoint"",""coordinates"":[]}",13,14,-1


In [ ]:
#visually testing different dataframes

In [68]:
filtered_train_df3.head(10)

,system:index,NIR,SWIR,blue,cloud_score,cyanobacteria_abundance,green,latitude,longitude,ndci,...,ndwi,red,red_edge,sample_date,satellite_date,.geo,tot_diff_days,mean_diff_days,median_diff_days,std_diff_days
0,9062_20160117T182712_20160117T183427_T11SMT,0.117157,0.197307,0.068217,0.072852,3824.5,0.086842,33.474200,-117.137998,0.302050,...,-0.246461,0.101563,0.159807,2016-01-21,2016-01-17,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409
1,9062_20151231T183752_20151231T183755_T11SMT,0.118239,0.181915,0.073154,0.000228,3824.5,0.094558,33.474200,-117.137998,0.270032,...,-0.209679,0.108157,0.153717,2016-01-21,2015-12-31,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409
2,9062_20151228T183312_20151228T183309_T11SMT,0.138406,0.206684,0.087906,0.072584,3824.5,0.108789,33.474200,-117.137998,0.275410,...,-0.193040,0.119495,0.181531,2016-01-21,2015-12-28,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409
3,9062_20151218T182802_20151218T182756_T11SMT,0.126158,0.202519,0.072346,0.000702,3824.5,0.094635,33.474200,-117.137998,0.345760,...,-0.253196,0.106488,0.180982,2016-01-21,2015-12-18,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409
4,9062_20151208T183312_20151208T183309_T11SMT,0.129843,0.200796,0.070884,0.038293,3824.5,0.094722,33.474200,-117.137998,0.361775,...,-0.242766,0.105682,0.189615,2016-01-21,2015-12-08,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409
5,9062_20151121T184422_20151121T184425_T11SMT,0.128463,0.192753,0.074177,0.000007,3824.5,0.098254,33.474200,-117.137998,0.374142,...,-0.241988,0.107506,0.188575,2016-01-21,2015-11-21,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409
6,9062_20151111T184002_20151111T183956_T11SMT,0.122889,0.180506,0.078283,0.012064,3824.5,0.103311,33.474200,-117.137998,0.404530,...,-0.200453,0.106544,0.196478,2016-01-21,2015-11-11,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409
7,9063_20160117T182712_20160117T183427_T11SMT,0.098329,0.216473,0.031464,0.072852,5943.0,0.053812,33.477901,-117.141998,0.401405,...,-0.299300,0.069213,0.163292,2016-01-21,2016-01-17,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409
8,9063_20151231T183752_20151231T183755_T11SMT,0.083996,0.193533,0.023260,0.000228,5943.0,0.039784,33.477901,-117.141998,0.406114,...,-0.366930,0.057544,0.134886,2016-01-21,2015-12-31,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409
9,9063_20151228T183312_20151228T183309_T11SMT,0.098287,0.218209,0.035785,0.072584,5943.0,0.053873,33.477901,-117.141998,0.389605,...,-0.298316,0.069005,0.156130,2016-01-21,2015-12-28,"{""type"":""MultiPoint"",""coordinates"":[]}",71.0,11.166667,10.0,5.269409


In [69]:
#creating csv files for each one

#before files
filtered_train_df3.to_csv('before_30_7', index = False)
filtered_train_df2.to_csv('before_50_8', index = False) 
filtered_train_df1.to_csv('before_70_9', index = False)

#after files
filtered_test_df3.to_csv('after_30', index = False)
filtered_test_df2.to_csv('after_50', index = False)
filtered_test_df1.to_csv('after_70', index = False)

In [70]:
#make note of everything in the google docs --> exact shape of everything... where it comes from 
#FINAL versions of train and test data for our model --> 3 different sets of features
print(filtered_train_df1.shape)
print(filtered_train_df2.shape)
print(filtered_train_df3.shape)
print(filtered_test_df1.shape)
print(filtered_test_df2.shape)
print(filtered_test_df3.shape)

(48402, 22)
(41040, 22)
(30499, 22)
(21512, 21)
(20520, 21)
(17428, 21)
